# GNN Floor Plan Inference
Run GNN + Classical pipeline on a DXF. GPU recommended → Runtime → Change runtime type → T4 GPU.

In [ ]:
# ── 1. Upload scripts zip ─────────────────────────────────────────────────────
# On your machine, run:
#   cd /home/sai-ashish-mishra/Documents/Visionaries_Project
#   zip scripts.zip fpgraph.py gnninfer.py gnnmodel.py gnndata.py gnneval.py \
#       cvpipeline.py cvadapter.py cvexport.py indlabel.py tracka_core.py config.json
# Then upload that zip here.
from google.colab import files
print('Upload scripts.zip')
up = files.upload()
import zipfile
with zipfile.ZipFile(list(up.keys())[0]) as z:
    z.extractall('.')
print('Scripts extracted')

In [ ]:
# ── 2. Install deps ───────────────────────────────────────────────────────────
# shapely, scipy, pandas, numpy, torch already in Colab — only install missing ones
!pip install -q ezdxf svgelements huggingface_hub python-dotenv

In [ ]:
# ── 3. Download model weights ─────────────────────────────────────────────────
# Paste your HF_TOKEN here (from .env)
HF_TOKEN = 'hf_YOUR_TOKEN_HERE'

import os, shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

os.makedirs('models/gnn', exist_ok=True)
for fname in ['finetune/best.pt', 'finetune/head2_threshold.json']:
    src = hf_hub_download('saiashish29/GNN-CV', fname, repo_type='model', token=HF_TOKEN)
    dst = f'models/gnn/{Path(fname).name}'
    shutil.copy(src, dst)
    print(f'Downloaded {dst}')

In [ ]:
# ── 4. Upload your DXF ───────────────────────────────────────────────────────
from google.colab import files
print('Upload your .dxf file')
up = files.upload()
DXF_PATH = list(up.keys())[0]
print(f'Using: {DXF_PATH}')

In [ ]:
# ── 5. Classical extraction ───────────────────────────────────────────────────
import tracka_core, json

cfg = {**tracka_core.CONFIG, 'floor_block': None}
result = tracka_core.run(DXF_PATH, cfg)
classical = tracka_core.build_manifest(DXF_PATH, result, cfg)
print(f"Classical: {classical['summary']['rooms']} rooms, {classical['summary']['doors']} doors")

In [ ]:
# ── 6. GNN inference ─────────────────────────────────────────────────────────
import time, torch, json, fpgraph, gnninfer, cvpipeline
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

phase_a = json.loads(Path('config.json').read_text())
model, _ = gnninfer.load_model('models/gnn/best.pt', device)
try:
    model = torch.compile(model, mode='reduce-overhead')
except Exception:
    pass

lut   = fpgraph.build_label_map(phase_a['raw_class_names'], phase_a['harmonization'], phase_a['targets'])
stuff = fpgraph.stuff_mask(phase_a['targets'])
thr   = gnninfer.read_threshold('models/gnn/head2_threshold.json')

t0 = time.time()
gnn_manifest = cvpipeline.run_floor(
    DXF_PATH, model, phase_a, lut, stuff, device,
    work_dir='gnn_work', thr=thr,
    floor_block=classical.get('floor', {}).get('block'),
    amp=(device.type == 'cuda'),
    node_budget=32000, max_tiles=16
)
print(f"GNN done in {time.time()-t0:.1f}s")
print(f"GNN: {gnn_manifest['summary']['rooms']} rooms, {gnn_manifest['summary']['doors']} doors")

In [ ]:
# ── 7. Save & download manifests ─────────────────────────────────────────────
Path('manifest_classical.json').write_text(json.dumps(classical,    indent=1, default=float))
Path('manifest_gnn.json').write_text(     json.dumps(gnn_manifest, indent=1, default=float))

from google.colab import files
files.download('manifest_classical.json')
files.download('manifest_gnn.json')